# Root Finding for a Cubic Polynomial

Before building the Vitis HLS IP, we build a python simulation of the cubic root finder.  Unlike the prior lab which worked in fixed point, all the computations in this lab will be in single precision floating point.  The cubic function is as follows:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from numpy.typing import NDArray

def fcubic(
    x: NDArray[np.float32],
    a: NDArray[np.float32] 
) -> NDArray[np.float32]:
    """
    Compute the monic cubic function 
    
    a[0] + a[1]*x + a[2]*x^2 + x^3.

    Parameters
    ----------
    x : NDArray[np.float32]  shape (nx)
        The input array of x values.
    a : NDArray[np.float32]  shape (nx,deg) with deg = 3
        The coefficients of the cubic function. 
        a[i,j] is the coefficient of x[i]^j

    Returns
    -------
    y : NDArray[np.float32]  shape (nx)
        The output array of y values. 
    """
    y = a[:,0] + a[:,1]*x + a[:,2]*x**2 + x**3
    return y


We plot an example function 


In [ ]:
asing = np.array([-1.0, 0.5, 1.5], dtype=np.float32)
xplot = np.linspace(-2, 2, 100, dtype=np.float32)
yplot = fcubic(xplot, asing[None,:])

plt.plot(xplot, yplot)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Monic Cubic Function')
plt.grid()
plt.show()

We now build a simple root solver using the update algorithm:

```
x[k+1] = x[k] + step*f(x[k])
```

for some step size, `step`.  This algorithm will converge to a local root provided `f(x)` is increasing.  The function `f(x)` will be increasing in our case if `a[1]` and `a[2] >= 0`.  Complete the function below to perform the root finding.  The algorithm stops either when a maximum number of iterations is hit or if `|f(x[k])| < tol` for some tolerance.

In [ ]:
def fsolve(
    a: NDArray[np.float32], 
    x0: np.float32 = 0.0, 
    tol: np.float32 = 1e-6, 
    max_iter: int = 1000,
    step: np.float32 = 0.01
) -> NDArray[np.float32]:
    """
    Solve for the roots of the monic cubic function using Newton's method.

    Parameters
    ----------
    a : NDArray[np.float32]  shape (deg,) with deg = 3
        The coefficients of the cubic function. 
        a[i] is the coefficient of x^i
    x0 : np.float32, optional
        The initial guess for the roots. Default is 0.0.
    tol : np.float32, optional
        The tolerance for convergence. Default is 1e-6.
    max_iter : int, optional
        The maximum number of iterations. Default is 1000.
    step : np.float32, optional
        The step size for the update. Default is 0.01.

    Returns
    -------
    x : np.float32
        The estimated root of the cubic function.
    fx : np.float32
        The value of the cubic function at the estimated root.
    hist : dictionary
        A dictionary containing the history of the iterations, including:
        - 'x': list of x values at each iteration
        - 'f': list of f(x) values at each iteration
    """
    x = x0
    hist = {'x': [], 'f': []}
    
    for i in range(max_iter):

        # TODO:  Implement the Euler update steps
        #   fx = ...
        #   x = ...
        
        # TODO:  Save the results in the history
        #   hist['x'].append(... )
        #   hist['f'].append(... )
        
        # TODO:  Check for early termination when |fx| < tol
        
    return x, fx, hist

Now we can run the code on the polynomial above.  You see that the value of `|f(x)|` decreases but may get stuck at around `1e-5` to `1e-6`.  The reason is that `np.float32` has a finite precision and eventually we cannot get better root finding than this value.

In [ ]:
# Run fsolve on the parameters asing
x_root, fx, hist = fsolve(asing, x0=0.0, step=0.01, max_iter=1000, tol=1e-6)
print(f"Root found: x = {x_root}")
print(f"Function value at root: f(x) = {fx}")
print(f"Total iterations: {len(hist['x'])}")




In [ ]:
# Create side-by-side subplots for convergence history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# TODO:  Plot x values from hist['x'] vs. iteration number


# TODO:  Plot |f(x)| from hist['f'] vs. iteration number
# Use semilogy (log scale)

plt.tight_layout()

# Create test_outputs directory and save figure
import os
os.makedirs('test_outputs', exist_ok=True)
plt.savefig('test_outputs/fsolve_hist.png', dpi=150, bbox_inches='tight')
print("Figure saved to test_outputs/fsolve_hist.png")

plt.show()


## Generate Test Vectors

We now generate a set of test vectors that we can compare the Vitis HLS model against.

In [ ]:
test_inputs = [
    {'x0': 0.0, 'a0': -1.0, 'a1': 0.5, 'a2': 1.5, 'max_iter': 100, 'tol': 1e-6, 'step': 0.1},
    {'x0': 1.0, 'a0': 2.0, 'a1': 1.0, 'a2': 0.8, 'max_iter': 100, 'tol': 1e-6, 'step': 0.1},
    {'x0': -1.5, 'a0': -0.5, 'a1': 2.0, 'a2': 1.2, 'max_iter': 100, 'tol': 1e-6, 'step': 0.1}
]

# Run fsolve for each test vector
results = []
for i, test in enumerate(test_inputs):
    a = np.array([test['a0'], test['a1'], test['a2']], dtype=np.float32)
    x_root, fx, hist = fsolve(a, x0=test['x0'], step=test['step'], max_iter=test['max_iter'], tol=test['tol'])
    
    results.append({
        'a0': test['a0'],
        'a1': test['a1'],
        'a2': test['a2'],
        'x0': test['x0'],
        'step' : test['step'],
        'tol' : test['tol'],
        'max_iter' : test['max_iter'],
        'x_root': x_root,
        'fx': fx,
        'iterations': len(hist['x'])
    })
    
    print(f"Test {i+1}: Initial x0={test['x0']:.2f}, a=[{test['a0']:.2f}, {test['a1']:.2f}, {test['a2']:.2f}]")
    print(f"         Root found: x={x_root:.6f}, f(x)={fx:.6f}, Iterations: {len(hist['x'])}")
    print()


In [ ]:
# Create pandas dataframe from results
df = pd.DataFrame(results)
print("Results dataframe:")
print(df)
print()

# Save to CSV
import os
os.makedirs('test_outputs', exist_ok=True)
df.to_csv('test_outputs/tv_python.csv', index=False)
print("Results saved to test_outputs/tv_python.csv")
